# Multimodal RAG — Linked Text, Table & Image Collections

## Architecture Overview

```
PDF / Images / CSVs
        │
        ▼
┌────────────────────────────────────┐
│  Stage 1: Per-Page Document Parser │
│  ─ PyMuPDF → text blocks           │
│  ─ PyMuPDF → tables (find_tables)  │
│  ─ PyMuPDF → images → DeepSeek OCR│
└────────────────┬───────────────────┘
                 │  page_id links all 3 types
        ┌────────┼────────┐
        ▼        ▼        ▼
  [Text DB] [Table DB] [Image DB]   ← 3 ChromaDB collections
  qwen3 embeddings on all 3
        └────────┬────────┘
                 │ Linked retrieval by page_id
                 ▼
         qwen3.5:122b LLM
                 │
                 ▼
           Final Answer
```

**Key design:** Every chunk stores a `page_id`. When retrieving, the top-k text hits
fetch their `page_id`s, which are used to also pull the matching tables and images — 
so the LLM always gets the full picture from the same page.


In [ ]:
!pip install PyMuPDF langchain-text-splitters langchain-ollama langchain-community langchain-core chromadb pillow requests

: 

In [6]:
import os, shutil, base64, uuid, fitz
from pathlib import Path
from PIL import Image
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import chromadb

# ── Model Config ────────────────────────────────────────────
PDF_PATH       = "2025舞光LED21st(單頁水印可搜尋).pdf"
IMAGE_OUT_DIR  = "./extracted_images"   # temp folder for page images

EMBEDDING_MODEL = "qwen3-embedding"              # qwen3 embedding model in Ollama
VISION_MODEL    = "deepseek-ocr"       # multimodal OCR model
CHAT_MODEL      = "qwen3.5:122b"       # final LLM for answering

PERSIST_BASE    = "./multimodal_linked_db"
TEXT_PERSIST    = f"{PERSIST_BASE}/text"
TABLE_PERSIST   = f"{PERSIST_BASE}/table"
IMAGE_PERSIST   = f"{PERSIST_BASE}/image"

print("Config loaded.")
print(f"  Embedding : {EMBEDDING_MODEL}")
print(f"  Vision    : {VISION_MODEL}")
print(f"  Chat LLM  : {CHAT_MODEL}")


Config loaded.
  Embedding : qwen3-embedding
  Vision    : deepseek-ocr
  Chat LLM  : qwen3.5:122b


In [2]:
def deepseek_ocr(image_path: str) -> str:
    """Extract text/tables from an image using DeepSeek OCR via LangChain ChatOllama."""
    llm_vision = ChatOllama(model=VISION_MODEL, temperature=0)
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode("utf-8")
    ext = image_path.split(".")[-1].lower()
    mime = "image/jpeg" if ext in ["jpg", "jpeg"] else f"image/{ext}"
    message = HumanMessage(
        content=[
            {"type": "text", "text": (
                "Extract ALL text and tabular data from this image. "
                "Format any tables as Markdown tables. "
                "Return only the extracted content, nothing else."
            )},
            {"type": "image_url", "image_url": f"data:{mime};base64,{img_b64}"}
        ]
    )
    try:
        response = llm_vision.invoke([message])
        return response.content.strip()
    except Exception as e:
        print(f"  [OCR Error] {image_path}: {e}")
        return ""


In [3]:
def parse_pdf_to_linked_docs(pdf_path: str, image_out_dir: str):
    """
    Parses a PDF page by page, returning 3 lists of LangChain Documents:
      text_docs   — plain text per page
      table_docs  — markdown tables per page
      image_docs  — OCR-extracted image text per page

    Each Document has metadata: { page_id, page_num, source_type }
    allowing cross-collection linking during retrieval.
    """
    os.makedirs(image_out_dir, exist_ok=True)
    doc = fitz.open(pdf_path)

    text_docs, table_docs, image_docs = [], [], []

    for page_num, page in enumerate(doc):
        page_id = f"page_{page_num:04d}"  # e.g. "page_0001"

        # ── 1. Extract Plain Text ────────────────────────────
        raw_text = page.get_text("text").strip()
        if raw_text:
            text_docs.append(Document(
                page_content=raw_text,
                metadata={"page_id": page_id, "page_num": page_num, "source_type": "text"}
            ))

        # ── 2. Extract Tables ────────────────────────────────
        try:
            tables = page.find_tables()
            for t_idx, table in enumerate(tables):
                # Convert to markdown
                rows = table.extract()
                if not rows:
                    continue
                headers = rows[0]
                sep = ["---"] * len(headers)
                md_rows = [" | ".join(str(c) if c else "" for c in headers)]
                md_rows.append(" | ".join(sep))
                for row in rows[1:]:
                    md_rows.append(" | ".join(str(c) if c else "" for c in row))
                md_table = "\n".join(md_rows)

                table_docs.append(Document(
                    page_content=md_table,
                    metadata={"page_id": page_id, "page_num": page_num,
                               "table_index": t_idx, "source_type": "table"}
                ))
        except Exception as e:
            print(f"  [Table parse error] Page {page_num}: {e}")

        # ── 3. Extract Images → DeepSeek OCR ─────────────────
        image_list = page.get_images(full=True)
        for img_idx, img_info in enumerate(image_list):
            xref = img_info[0]
            base_image = doc.extract_image(xref)
            img_bytes = base_image["image"]
            img_ext = base_image.get("ext", "png")
            img_path = os.path.join(image_out_dir, f"{page_id}_img{img_idx}.{img_ext}")

            with open(img_path, "wb") as f:
                f.write(img_bytes)

            print(f"  → OCR on page {page_num} image {img_idx}...")
            ocr_text = deepseek_ocr(img_path)
            if ocr_text:
                image_docs.append(Document(
                    page_content=ocr_text,
                    metadata={"page_id": page_id, "page_num": page_num,
                               "image_path": img_path, "source_type": "image"}
                ))

    doc.close()
    print(f"\nParsing complete.")
    print(f"  Text docs  : {len(text_docs)}")
    print(f"  Table docs : {len(table_docs)}")
    print(f"  Image docs : {len(image_docs)}")
    return text_docs, table_docs, image_docs

# ── Run Parser ──────────────────────────────────────────────
print(f"Parsing PDF: {PDF_PATH}")
text_docs, table_docs, image_docs = parse_pdf_to_linked_docs(PDF_PATH, IMAGE_OUT_DIR)


Parsing PDF: 2025舞光LED21st(單頁水印可搜尋).pdf
Consider using the pymupdf_layout package for a greatly improved page layout analysis.
  → OCR on page 0 image 0...
  → OCR on page 1 image 0...
  → OCR on page 1 image 1...
  → OCR on page 1 image 2...
  → OCR on page 1 image 3...
  → OCR on page 1 image 4...
  → OCR on page 1 image 5...
  → OCR on page 1 image 6...
  → OCR on page 1 image 7...
  → OCR on page 1 image 8...
  → OCR on page 1 image 9...
  → OCR on page 1 image 10...
  → OCR on page 1 image 11...
  → OCR on page 1 image 12...
  → OCR on page 1 image 13...
  → OCR on page 1 image 14...
  → OCR on page 1 image 15...
  → OCR on page 1 image 16...
  → OCR on page 1 image 17...
  → OCR on page 1 image 18...
  → OCR on page 1 image 19...
  → OCR on page 1 image 20...
  → OCR on page 1 image 21...
  → OCR on page 1 image 22...
  → OCR on page 2 image 0...
  → OCR on page 2 image 1...
  → OCR on page 2 image 2...
  → OCR on page 3 image 0...
  → OCR on page 3 image 1...
  → OCR on page 3 i

In [7]:
# ── Clear old DBs ───────────────────────────────────────────
chromadb.api.client.SharedSystemClient.clear_system_cache()
for d in [TEXT_PERSIST, TABLE_PERSIST, IMAGE_PERSIST]:
    if os.path.exists(d):
        shutil.rmtree(d)

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)

# ── Build 3 linked Chroma collections ───────────────────────
print("Building Text DB...")
text_db = Chroma.from_documents(text_docs, embeddings, persist_directory=TEXT_PERSIST)

print("Building Table DB...")
if table_docs:
    table_db = Chroma.from_documents(table_docs, embeddings, persist_directory=TABLE_PERSIST)
else:
    table_db = Chroma(persist_directory=TABLE_PERSIST, embedding_function=embeddings)
    print("  (no tables found)")

print("Building Image DB...")
if image_docs:
    image_db = Chroma.from_documents(image_docs, embeddings, persist_directory=IMAGE_PERSIST)
else:
    image_db = Chroma(persist_directory=IMAGE_PERSIST, embedding_function=embeddings)
    print("  (no image OCR content found)")

print("\n✅ All 3 databases built and linked by page_id.")


Building Text DB...
Building Table DB...
  (no tables found)
Building Image DB...


/tmp/ipykernel_312356/2131418333.py:17: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  table_db = Chroma(persist_directory=TABLE_PERSIST, embedding_function=embeddings)



✅ All 3 databases built and linked by page_id.


In [8]:
def linked_retrieval(query: str, top_k: int = 3) -> dict:
    """
    1. Query text_db for top_k results.
    2. Collect the page_ids from those hits.
    3. Fetch matching tables and images from page_ids.
    4. Return combined context dict.
    """
    # ── Step 1: Semantic search on text collection ────────────
    text_retriever = text_db.as_retriever(search_kwargs={"k": top_k})
    text_hits = text_retriever.invoke(query)

    # ── Step 2: Gather page_ids ───────────────────────────────
    page_ids = list({doc.metadata["page_id"] for doc in text_hits})
    print(f"  Matched page_ids: {page_ids}")

    # ── Step 3: Fetch tables for those page_ids ────────────────
    table_hits = []
    try:
        table_results = table_db._collection.get(
            where={"page_id": {"$in": page_ids}},
            include=["documents", "metadatas"]
        )
        for i, content in enumerate(table_results["documents"]):
            table_hits.append(Document(
                page_content=content,
                metadata=table_results["metadatas"][i]
            ))
    except Exception as e:
        print(f"  [Table retrieval] {e}")

    # ── Step 4: Fetch images for those page_ids ────────────────
    image_hits = []
    try:
        image_results = image_db._collection.get(
            where={"page_id": {"$in": page_ids}},
            include=["documents", "metadatas"]
        )
        for i, content in enumerate(image_results["documents"]):
            image_hits.append(Document(
                page_content=content,
                metadata=image_results["metadatas"][i]
            ))
    except Exception as e:
        print(f"  [Image retrieval] {e}")

    return {
        "text_hits":  text_hits,
        "table_hits": table_hits,
        "image_hits": image_hits,
        "page_ids":   page_ids,
    }

print("Linked retrieval function ready.")


Linked retrieval function ready.


In [9]:
def build_context(retrieved: dict) -> str:
    """Merge text, table and image context into a single prompt string."""
    parts = []

    if retrieved["text_hits"]:
        parts.append("### 📄 Text Context")
        parts.extend(doc.page_content for doc in retrieved["text_hits"])

    if retrieved["table_hits"]:
        parts.append("\n### 📊 Table Context")
        parts.extend(doc.page_content for doc in retrieved["table_hits"])

    if retrieved["image_hits"]:
        parts.append("\n### 🖼️ Image OCR Context")
        parts.extend(doc.page_content for doc in retrieved["image_hits"])

    return "\n\n".join(parts)


PROMPT_TEMPLATE = """
### 角色定位
你是一位專業的「舞光 (DanceLight)」LED 照明產品顧問。
你同時擁有來自產品目錄的文字描述、規格表格、以及圖片辨識內容，請綜合三種資料來回答。

### 檢索到的內容 (Text + Tables + Image OCR)
{context}

### 任務指令
1. 分析使用者需求，從文字、表格、圖片三種資料中提取相關訊息。
2. 推薦產品時，列出：型號、瓦數、色溫、演色性、特殊功能。
3. 使用「列點」格式，讓規格一目了然。

### 回答限制
- 僅根據上方提供的內容回答。
- 嚴禁自行編造產品型號或規格。
- 一律使用「繁體中文」回答。

---
使用者問題：{question}
顧問建議：
"""
prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

llm = ChatOllama(
    model=CHAT_MODEL,
    temperature=0,
    num_gpu=99,
    keep_alive="30m"
)

def rag_answer(question: str, top_k: int = 3):
    print(f"\nQuestion: {question}")
    print(f"[{CHAT_MODEL} 檢索與思考中...]\n")
    
    retrieved = linked_retrieval(question, top_k=top_k)
    context_str = build_context(retrieved)
    
    chain = prompt | llm | StrOutputParser()
    
    print("\n--- 回答 ---\n")
    for chunk in chain.stream({"context": context_str, "question": question}):
        print(chunk, end="", flush=True)
    print("\n")

print("RAG chain ready.")


RAG chain ready.


In [14]:
# ── Test Query ──────────────────────────────────────────────
rag_answer("車頭燈有賣嗎")



Question: 車頭燈有賣嗎
[qwen3.5:122b 檢索與思考中...]

  Matched page_ids: ['page_0039', 'page_0121', 'page_0006']

--- 回答 ---

您好，我是舞光 (DanceLight) LED 照明產品顧問。

針對您詢問的「車頭燈」，經檢索提供的產品目錄內容，**目前資料中並無此類汽車照明產品**。

根據現有目錄，我們主要提供以下建築與室內照明類別（非汽車用燈）：

*   **戶外照明系列**：防水壁燈、照樹燈、洗柱燈、插地燈、地底燈、步道燈。
*   **室內/商業照明系列**：神盾吸頂筒燈系列、邱比特軌道燈系列、商業用燈泡、居家燈泡、燈絲燈、燈管。
*   **光源與配件系列**：AR111 模組、MR16 杯燈、防水接頭、軌道、感應器、緊急照明燈、驅動器。

由於提供的資料中無車頭燈之相關資訊，因此**無法提供該項目的型號、瓦數、色溫、演色性及特殊功能等規格**。

若您有戶外景觀或建築照明需求（如步道燈、照樹燈等），歡迎進一步詢問，我將為您提供對應的產品規格。

